In [ ]:
import requests
from bs4 import BeautifulSoup
import sqlite3
import pandas as pd
from datetime import datetime

current_year = datetime.now().year
current_month = datetime.now().month

# 크롤링할 사이트별 메서드
def crawl_site_lukina(site_name, base_url):
    data = []
    
    
    
    # 12개월 동안 크롤링
    for month in range(current_month, current_month + 12):
        year = current_year
        if month > 12:
            month = month - 12
            year = current_year + 1
        
        # URL 생성
        url = f"{base_url}/{year}{month:02d}"
        response = requests.get(url)
        html = response.text
        soup = BeautifulSoup(html, 'html.parser')
        
        # 날짜 정보 추출
        items = soup.select(".shipsinfo_daywarp")
        
        if not items:
            break
        
        # 날짜별 데이터 추출
        for item in items:
            day_part_raw = item.select_one(".date_info").text.strip().replace("\n", "")
            cut_index = day_part_raw.find(")") + 1
            date = day_part_raw[:cut_index]
            
            date_final = f"{year}년 {date}"
            wave_power = item.select_one(".date_info2").text.strip().replace('\n', '')
            zone = "인천권"
            ships = item.select(".small_event_wrap")
            for ship in ships:
                ship_name = ship.select_one(".ship_info>.title").text.strip()
                try:
                    fish_name = ship.select_one(".fishspecies")
                    if fish_name:
                        fish_name = fish_name.text.replace("어종 :", "").split("/")[0].strip()
                       
                    else:
                        fish_name ='[]'
                except AttributeError:
                 print([])  # 예외 발생 시 빈 리스트 출력
                
                reservation_element = ship.select_one(".number.blink_me.n_blue.f_20")
                reservation = reservation_element.text.strip() if reservation_element else "마감"
                booking_url = f"{base_url}/{year}{month:02d}"
                
                data.append([zone,site_name, ship_name, date_final, wave_power, fish_name, reservation, booking_url])
    df = pd.DataFrame(data, columns=['지역','사이트', '선박명', '날짜', '조류세기', '어종', '예약자리','바로가기'])
    return data

# 다른 사이트용 크롤링 메서드 (예시)
def crawl_site_rise(site_name, base_url):
    data = []
   

    site_name = "영종도라이즈호"
    base_url = "https://risefishing.sunsang24.com/ship/schedule_fleet"  # 실제 사이트 URL로 변경하세요.

    

    # 12개월 동안 크롤링
    for month in range(current_month, current_month + 12):
        year = current_year
        if month > 12:
            month = month - 12
            year = current_year + 1
        
        # URL 생성
        url = f"{base_url}/{year}{month:02d}"
        response = requests.get(url)
        html = response.text
        soup = BeautifulSoup(html, 'html.parser')
        items = soup.select(".shipsinfo_daywarp")
        if not items:
            break
        # 날짜 추출
        for item in items:
            # 선박명
            ship_name = item.select_one(".ship_info>div.title").text.strip()
            zone = "인천권"
            # 날짜 및 조류세기
            
            day_part_raw = item.select_one(".date_wrap").text.strip().replace("\n", "")
            cut_index = day_part_raw.find(")") + 1
            date = day_part_raw[:cut_index]
            date_final = f"{year}년 {date}"
            
            wave_power = item.select_one(".date_info2").text.strip()

            # 어종 정보
            fish_name = item.select_one(".fishspecies").text.replace("어종 :", "").split("/")[0].strip()

            # 예약 정보
            reser_img = item.select_one('li.remain').text.strip()
            if "남은자리" in reser_img:
                reservation = item.select_one('li.remain span.number').text.strip()
            else:
                reservation = "마감"
            booking_url = f"{base_url}/{year}{month:02d}"    
            
            # 데이터 리스트에 추가
            data.append([zone,site_name, ship_name, date_final, wave_power, fish_name, reservation, booking_url])
    df = pd.DataFrame(data, columns=['지역','사이트', '선박명', '날짜', '조류세기', '어종', '예약자리','바로가기'])
    return data

def crawl_site_ottogi(site_name, base_url):
   
    site_name = "영종도오뚜기호"
    base_url = "http://ottogi.sunsang24.com/ship/schedule_fleet"

    data = []

    # 12개월 동안 크롤링
    for month in range(current_month, current_month + 12):
        year = current_year
        if month > 12:
            month = month - 12
            year = current_year + 1
        
        # URL 생성
        url = f"{base_url}/{year}{month:02d}"
        response = requests.get(url)
        html = response.text
        soup = BeautifulSoup(html, 'html.parser')
        items = soup.select(".shipsinfo_daywarp")
      
        # 날짜 추출
        for item in items:
        # 날짜 및 조류세기
            
            day_part_raw = item.select_one(".date_wrap").text.strip().replace("\n", "")
            cut_index = day_part_raw.find(")") + 1
            date = day_part_raw[:cut_index]
            date_final = f"{year}년 {date}"
            wave_power = item.select_one(".date_info2").text.strip()
            zone = "인천권"
            
            ships = item.select(".ships_warp>table")
            for ship in ships:
                ship_name = ship.select_one(".ship_info>div.title").text.strip()
                fish_element = ship.select_one("#fish")
                if fish_element is not None:
                    fish_name = fish_element.text.strip().replace('[', '').replace(']', '')
                else:
                    fish_name = '[]'
                reservation = ship.select_one('.remain').text.strip()
                if "남은자리" in reservation:
                    reservation = ship.select_one('.number.blink_me').text.strip()
                else:
                    reservation = "마감"
                booking_url = f"{base_url}/{year}{month:02d}"  
                    
                data.append([zone,site_name, ship_name, date_final, wave_power, fish_name, reservation, booking_url])

    # DataFrame 생성
    df = pd.DataFrame(data, columns=['지역','사이트', '선박명', '날짜', '조류세기', '어종', '예약자리','바로가기'])
    return data
    
# 여러 사이트 크롤링 후 DB에 저장
def crawl_and_save_to_db():
    # 데이터베이스 연결
    conn = sqlite3.connect('cruise_schedule.db')
    cursor = conn.cursor()

    # 테이블 생성 (만약 없으면)
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS cruise_schedule (
            zone TEXT,
            site_name TEXT,
            ship_name TEXT,
            date TEXT,
            wave_power TEXT,
            fish_name TEXT,
            reservation TEXT,
            booking_url TEXT
        )
    ''')

    # 크롤링할 사이트 목록과 URL
    sites = {
        "루키나호": "https://lukina.sunsang24.com/ship/schedule_fleet",
        "영종도라이즈호": "https://risefishing.sunsang24.com/ship/schedule_fleet",
        "영종도오뚜기호": "http://ottogi.sunsang24.com/ship/schedule_fleet"
    }

    for site_name, base_url in sites.items():
    # 기존 사이트 데이터 삭제 (덮어쓰기)
        cursor.execute('DELETE FROM cruise_schedule WHERE site_name = ?', (site_name,))
        
        if site_name == "루키나호":
            site_data = crawl_site_lukina(site_name, base_url)
        elif site_name == "영종도라이즈호":
            site_data = crawl_site_rise(site_name, base_url)
        elif site_name == "영종도오뚜기호":
            site_data = crawl_site_ottogi(site_name, base_url)    
            
            
        
        # 크롤링된 데이터를 DB에 삽입
        for row in site_data:
            cursor.execute('''
                INSERT INTO cruise_schedule (zone,site_name, ship_name,date, wave_power, fish_name, reservation, booking_url)
                VALUES (?, ?, ?, ?, ?, ?, ?, ?)
            ''', row)
    
    # 커밋 후 데이터베이스 연결 종료
    conn.commit()
    conn.close()

    # DB에서 데이터 불러와 출력
    conn = sqlite3.connect('cruise_schedule.db')
    df = pd.read_sql('SELECT * FROM cruise_schedule', conn)
    conn.close()
    
    return df

# 크롤링 및 DB 저장 후 결과 출력
df = crawl_and_save_to_db()
print(df)


     zone site_name ship_name            date wave_power fish_name  \
0     인천권      루키나호    루키나 2호  2025년 5월12일(월)         6물        참돔   
1     인천권      루키나호      루키나호  2025년 5월12일(월)         6물        광어   
2     인천권      루키나호    루키나 5호  2025년 5월12일(월)         6물        광어   
3     인천권      루키나호    루키나 2호  2025년 5월13일(화)         7물        참돔   
4     인천권      루키나호      루키나호  2025년 5월13일(화)         7물        광어   
...   ...       ...       ...             ...        ...       ...   
2184  인천권   영종도오뚜기호       미르호  2026년 4월29일(수)         4물        []   
2185  인천권   영종도오뚜기호      비키니호  2026년 4월29일(수)         4물        []   
2186  인천권   영종도오뚜기호      오뚜기호  2026년 4월30일(목)         5물        []   
2187  인천권   영종도오뚜기호       미르호  2026년 4월30일(목)         5물        []   
2188  인천권   영종도오뚜기호      비키니호  2026년 4월30일(목)         5물        []   

     reservation                                        booking_url  
0             7명  https://lukina.sunsang24.com/ship/schedule_fle...  
1             4명  h